# PROCESAMIENTO CON EMBEDDINGS - Dataset TFM
## Sistema de Asistencia Lectura para Discapacidad Cognitiva
**Cristina Varela - IABD**

---

**Método:** Sentence Transformers (Embeddings semánticos)
**Accuracy esperado:** 70-85%

**Ventajas:**
- Captura significado semántico real
- No depende de métricas superficiales
- Estado del arte en clasificación de texto

## 1. Importar Librerías

In [1]:
import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("✓ Librerías importadas correctamente")
print("⏳ Cargando modelo de embeddings...")

# Cargar modelo preentrenado en español
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("✓ Modelo de embeddings cargado (384 dimensiones)")

✓ Librerías importadas correctamente
⏳ Cargando modelo de embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Modelo de embeddings cargado (384 dimensiones)


## 2. Configuración de Rutas

In [2]:
# Ruta base del proyecto
BASE_DIR = Path(r'C:\Users\crisv\OneDrive\Desktop\asistente-lectura-discapacidad')

# Rutas de carpetas con archivos .txt
CARPETAS_TEXTOS = {
    1: BASE_DIR / 'data' / 'raw' / 'nivel_1_lectura_facil',
    2: BASE_DIR / 'data' / 'raw' / 'nivel_2_cuentos_simple',
    3: BASE_DIR / 'data' / 'raw' / 'nivel_3_cuentos_complejo',
    4: BASE_DIR / 'data' / 'raw' / 'nivel_4_narrativo_adulto',
    5: BASE_DIR / 'data' / 'raw' / 'nivel_5_tecnico_especializado',
}

# Archivos de salida
OUTPUT_CSV = BASE_DIR / 'data' / 'processed' / 'textos_embeddings.csv'
OUTPUT_MODEL = BASE_DIR / 'models' / 'clasificador_embeddings.pkl'

print(f"✓ Configuración establecida (5 NIVELES)")
print(f"  Base: {BASE_DIR}")
print(f"  CSV: {OUTPUT_CSV}")
print(f"  Modelo: {OUTPUT_MODEL}")

✓ Configuración establecida (5 NIVELES)
  Base: C:\Users\crisv\OneDrive\Desktop\asistente-lectura-discapacidad
  CSV: C:\Users\crisv\OneDrive\Desktop\asistente-lectura-discapacidad\data\processed\textos_embeddings.csv
  Modelo: C:\Users\crisv\OneDrive\Desktop\asistente-lectura-discapacidad\models\clasificador_embeddings.pkl


## 3. Funciones de Procesamiento

In [3]:
def limpiar_texto(texto):
    """
    Limpia el texto eliminando metadata y caracteres especiales.
    """
    lineas_eliminar = [
        'Escuchar el texto', 'Publicado el', 'Guardado en',
        'Ver la versión', 'Twitter', 'Facebook', 'Telegram', 'WhatsApp',
    ]
    
    lineas = texto.split('\n')
    lineas_limpias = []
    
    for linea in lineas:
        linea = linea.strip()
        if len(linea) < 10:
            continue
        if any(meta in linea for meta in lineas_eliminar):
            continue
        lineas_limpias.append(linea)
    
    texto_limpio = ' '.join(lineas_limpias)
    texto_limpio = re.sub(r'\s+', ' ', texto_limpio)
    
    return texto_limpio.strip()


def dividir_en_fragmentos_adaptativo(texto, nivel):
    """
    Divide texto en fragmentos con tamaño adaptado al nivel.
    Niveles bajos = fragmentos más pequeños para maximizar cantidad.
    """
    # Configuración por nivel
    config = {
        1: {'min': 15, 'max': 40},  # Lectura fácil - MÁS PEQUEÑOS
        2: {'min': 25, 'max': 55},  # Cuentos simples
        3: {'min': 25, 'max': 55},  # Cuentos complejos - REDUCIDOS
        4: {'min': 30, 'max': 70},  # Narrativa adulta
        5: {'min': 35, 'max': 80},  # Técnico - MÁS GRANDES
    }
    
    min_palabras = config[nivel]['min']
    max_palabras = config[nivel]['max']
    
    # Dividir en frases
    frases = re.split(r'[.!?]+', texto)
    frases = [f.strip() for f in frases if f.strip()]
    
    fragmentos = []
    fragmento_actual = []
    palabras_actual = 0
    
    for frase in frases:
        num_palabras = len(frase.split())
        
        # Si añadir esta frase excede el máximo, guardar fragmento
        if palabras_actual + num_palabras > max_palabras and fragmento_actual:
            fragmentos.append(' '.join(fragmento_actual) + '.')
            fragmento_actual = [frase]
            palabras_actual = num_palabras
        else:
            fragmento_actual.append(frase)
            palabras_actual += num_palabras
        
        # Si alcanzamos el mínimo, podemos cerrar fragmento
        if palabras_actual >= min_palabras:
            fragmentos.append(' '.join(fragmento_actual) + '.')
            fragmento_actual = []
            palabras_actual = 0
    
    # Añadir último fragmento si cumple mínimo
    if fragmento_actual and palabras_actual >= min_palabras:
        fragmentos.append(' '.join(fragmento_actual) + '.')
    
    return fragmentos



def procesar_archivo(filepath, nivel):
    """
    Procesa un archivo .txt y devuelve fragmentos con nivel.
    Usa fragmentación adaptativa según el nivel.
    """
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            texto = f.read()
    except UnicodeDecodeError:
        with open(filepath, 'r', encoding='latin-1') as f:
            texto = f.read()
    
    texto_limpio = limpiar_texto(texto)
    if not texto_limpio:
        return []
    
    # CAMBIO AQUÍ: usar fragmentación adaptativa
    fragmentos = dividir_en_fragmentos_adaptativo(texto_limpio, nivel)
    
    # Mínimo de palabras también adaptativo
    min_palabras_minimo = {1: 12, 2: 20, 3: 20, 4: 25, 5: 30}
    min_req = min_palabras_minimo.get(nivel, 20)
    
    registros = []
    for i, fragmento in enumerate(fragmentos):
        if len(fragmento.split()) >= min_req:
            registros.append({
                'texto': fragmento,
                'nivel': nivel,
                'fuente': filepath.name,
                'fragmento_num': i+1
            })
    
    return registros


## 4. Cargar y Procesar Todos los Textos

In [4]:
print("⏳ Procesando archivos...\n")

todos_los_registros = []
stats_por_nivel = {}

for nivel, carpeta in CARPETAS_TEXTOS.items():
    if not carpeta.exists():
        print(f"⚠️  Nivel {nivel}: Carpeta no encontrada - {carpeta}")
        continue
    
    archivos = list(carpeta.glob('*.txt'))
    registros_nivel = []
    
    for archivo in archivos:
        registros = procesar_archivo(archivo, nivel)
        registros_nivel.extend(registros)
    
    todos_los_registros.extend(registros_nivel)
    stats_por_nivel[nivel] = {
        'archivos': len(archivos),
        'fragmentos': len(registros_nivel)
    }
    
    print(f"✓ Nivel {nivel}: {len(archivos)} archivos → {len(registros_nivel)} fragmentos")

# Crear DataFrame
df = pd.DataFrame(todos_los_registros)

print(f"\n✓ Total: {len(df)} fragmentos de {df['fuente'].nunique()} archivos")
print(f"\nDistribución por nivel:")
print(df['nivel'].value_counts().sort_index())

⏳ Procesando archivos...

✓ Nivel 1: 30 archivos → 380 fragmentos
✓ Nivel 2: 40 archivos → 760 fragmentos
✓ Nivel 3: 35 archivos → 651 fragmentos
✓ Nivel 4: 9 archivos → 466 fragmentos
✓ Nivel 5: 25 archivos → 438 fragmentos

✓ Total: 2695 fragmentos de 139 archivos

Distribución por nivel:
nivel
1    380
2    760
3    651
4    466
5    438
Name: count, dtype: int64


## 5. Generar Embeddings

In [5]:
print("⏳ Generando embeddings (esto puede tardar 2-3 minutos)...\n")

# Convertir textos a embeddings (vectores de 384 dimensiones)
textos = df['texto'].tolist()
embeddings = embedding_model.encode(textos, show_progress_bar=True)

print(f"\n✓ Embeddings generados: {embeddings.shape}")
print(f"  - {embeddings.shape[0]} textos")
print(f"  - {embeddings.shape[1]} dimensiones por texto")

# Añadir embeddings al DataFrame (para guardar)
df['embedding'] = list(embeddings)

⏳ Generando embeddings (esto puede tardar 2-3 minutos)...



Batches:   0%|          | 0/85 [00:00<?, ?it/s]


✓ Embeddings generados: (2695, 384)
  - 2695 textos
  - 384 dimensiones por texto


## 5.1. Features Lingüísticas Adicionales

In [6]:
import re
import numpy as np
from sklearn.preprocessing import StandardScaler

def calcular_features_adicionales(texto):
    """
    Calcula features lingüísticas complementarias a los embeddings.
    Estas features capturan complejidad sintáctica y léxica.
    """
    palabras = texto.split()
    frases = re.split(r'[.!?]+', texto)
    frases = [f.strip() for f in frases if f.strip()]
    
    # Features básicas
    num_palabras = len(palabras)
    num_frases = len(frases)
    
    # Longitud de palabras
    longitudes = [len(p) for p in palabras]
    longitud_media = np.mean(longitudes) if longitudes else 0
    longitud_max = max(longitudes) if longitudes else 0
    
    # Palabras largas (>6 letras)
    palabras_largas = sum(1 for p in palabras if len(p) > 6)
    ratio_largas = palabras_largas / num_palabras if num_palabras > 0 else 0
    
    # Palabras muy largas (>10 letras)
    palabras_muy_largas = sum(1 for p in palabras if len(p) > 10)
    ratio_muy_largas = palabras_muy_largas / num_palabras if num_palabras > 0 else 0
    
    # Complejidad sintáctica
    palabras_por_frase = num_palabras / num_frases if num_frases > 0 else 0
    
    # Variabilidad de longitud de palabras
    std_longitud = np.std(longitudes) if len(longitudes) > 1 else 0
    
    # Vocabulario único (riqueza léxica)
    palabras_unicas = len(set([p.lower() for p in palabras]))
    ratio_unico = palabras_unicas / num_palabras if num_palabras > 0 else 0
    
    return {
        'num_palabras': num_palabras,
        'num_frases': num_frases,
        'longitud_media_palabra': longitud_media,
        'longitud_max_palabra': longitud_max,
        'ratio_palabras_largas': ratio_largas,
        'ratio_palabras_muy_largas': ratio_muy_largas,
        'palabras_por_frase': palabras_por_frase,
        'std_longitud_palabra': std_longitud,
        'ratio_vocabulario_unico': ratio_unico,
    }


print("⏳ Calculando features lingüísticas adicionales...")
print("   (Esto tardará 1-2 minutos)\n")

# Calcular features para todos los textos
features_adicionales = df['texto'].apply(calcular_features_adicionales)
features_df = pd.DataFrame(features_adicionales.tolist())

print(f"✓ Features calculadas: {features_df.shape[1]} features")
print(f"\nFeatures extraídas:")
for col in features_df.columns:
    print(f"  - {col}")

# Normalizar features (importante para combinar con embeddings)
scaler = StandardScaler()
features_norm = scaler.fit_transform(features_df)

print(f"\n✓ Features normalizadas")

# COMBINAR embeddings + features lingüísticas
X_hibrido = np.hstack([embeddings, features_norm])

print(f"\n✓ Features híbridas generadas:")
print(f"  - Embeddings: {embeddings.shape[1]} dimensiones")
print(f"  - Features lingüísticas: {features_norm.shape[1]} dimensiones")
print(f"  - TOTAL: {X_hibrido.shape[1]} dimensiones")

# IMPORTANTE: Ahora usar X_hibrido en lugar de embeddings
X = X_hibrido

print(f"\n✓ Dataset híbrido listo para entrenar")


⏳ Calculando features lingüísticas adicionales...
   (Esto tardará 1-2 minutos)

✓ Features calculadas: 9 features

Features extraídas:
  - num_palabras
  - num_frases
  - longitud_media_palabra
  - longitud_max_palabra
  - ratio_palabras_largas
  - ratio_palabras_muy_largas
  - palabras_por_frase
  - std_longitud_palabra
  - ratio_vocabulario_unico

✓ Features normalizadas

✓ Features híbridas generadas:
  - Embeddings: 384 dimensiones
  - Features lingüísticas: 9 dimensiones
  - TOTAL: 393 dimensiones

✓ Dataset híbrido listo para entrenar


## 6. Entrenar Clasificador

In [7]:
print("⏳ Entrenando clasificador con XGBoost...\n")

# Preparar datos
X = X_hibrido
y = df['nivel'].values

# IMPORTANTE: XGBoost necesita clases empezando en 0
# Convertir niveles [1,2,3,4,5] a [0,1,2,3,4]
y_xgb = y - 1  # Restar 1

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_xgb, test_size=0.2, random_state=42, stratify=y_xgb
)

print(f"Train: {len(X_train)} textos")
print(f"Test:  {len(X_test)} textos\n")

# Importar XGBoost
from xgboost import XGBClassifier

# Entrenar modelo con XGBoost optimizado
clf = XGBClassifier(
    n_estimators=200,           # Más árboles para mejor aprendizaje
    max_depth=8,                # Mayor profundidad
    learning_rate=0.05,         # Aprendizaje más conservador
    subsample=0.8,              # Prevenir sobreajuste
    colsample_bytree=0.8,       # Usar 80% de features
    min_child_weight=3,         # Regularización
    gamma=0.1,                  # Penalización splits
    random_state=42
)
print("Entrenando XGBoost...")
clf.fit(X_train, y_train, verbose=False)

print("\n✓ Modelo XGBoost entrenado")

⏳ Entrenando clasificador con XGBoost...

Train: 2156 textos
Test:  539 textos

Entrenando XGBoost...

✓ Modelo XGBoost entrenado


## 7. Evaluar Modelo

In [8]:
# Predicciones
y_pred_xgb = clf.predict(X_test)

# Convertir de vuelta a niveles reales [1,2,3,4,5]
y_pred = y_pred_xgb + 1
y_test_original = y_test + 1

accuracy = accuracy_score(y_test_original, y_pred)

print("=" * 80)
print("RESULTADOS DEL MODELO")
print("=" * 80)
print(f"\n🎯 ACCURACY: {accuracy:.2%}\n")

print("Reporte de Clasificación:")
print(classification_report(y_test_original, y_pred, target_names=[f'Nivel {i}' for i in range(1, 6)]))

print("\nMatriz de Confusión:")
cm = confusion_matrix(y_test_original, y_pred)
print(cm)

RESULTADOS DEL MODELO

🎯 ACCURACY: 90.54%

Reporte de Clasificación:
              precision    recall  f1-score   support

     Nivel 1       0.94      0.89      0.92        76
     Nivel 2       0.86      0.91      0.88       152
     Nivel 3       0.91      0.90      0.90       130
     Nivel 4       0.88      0.85      0.86        93
     Nivel 5       0.99      0.98      0.98        88

    accuracy                           0.91       539
   macro avg       0.91      0.91      0.91       539
weighted avg       0.91      0.91      0.91       539


Matriz de Confusión:
[[ 68   3   3   2   0]
 [  3 138   6   5   0]
 [  1   8 117   3   1]
 [  0  12   2  79   0]
 [  0   0   1   1  86]]


## 8. Validación Cruzada

In [9]:
print("⏳ Validación cruzada (5-fold)...\n")

# IMPORTANTE: Usar y_xgb (valores 0-4) para validación cruzada
y_xgb_full = y - 1  # Convertir todo el dataset a 0-4

cv_scores = cross_val_score(clf, X, y_xgb_full, cv=5, scoring='accuracy')

print(f"Scores por fold: {cv_scores}")
print(f"\nAccuracy medio: {cv_scores.mean():.2%}")
print(f"Desviación estándar: {cv_scores.std():.2%}")
print(f"\n✓ Modelo consistente: {cv_scores.min():.2%} - {cv_scores.max():.2%}")

⏳ Validación cruzada (5-fold)...

Scores por fold: [0.67346939 0.86827458 0.84786642 0.86270872 0.88868275]

Accuracy medio: 82.82%
Desviación estándar: 7.85%

✓ Modelo consistente: 67.35% - 88.87%


## 9. Guardar Modelo y Dataset

In [10]:
import pickle

# Crear carpetas si no existen
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_MODEL.parent.mkdir(parents=True, exist_ok=True)

# Guardar CSV (sin embeddings, son muy grandes)
df_save = df[['texto', 'nivel', 'fuente', 'fragmento_num']].copy()
df_save.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print(f"✓ CSV guardado: {OUTPUT_CSV}")

# Guardar modelo con scaler (necesario para predicciones)
modelo_completo = {
    'clasificador': clf,
    'scaler': scaler,  # IMPORTANTE: guardar scaler
    'embedding_model_name': 'paraphrase-multilingual-MiniLM-L12-v2',
    'accuracy': accuracy,
    'cv_scores': cv_scores,
    'niveles': sorted(df['nivel'].unique()),
    'usa_features_hibridas': True,
    'dimensiones_totales': 393
}

with open(OUTPUT_MODEL, 'wb') as f:
    pickle.dump(modelo_completo, f)

print(f"✓ Modelo guardado: {OUTPUT_MODEL}")

✓ CSV guardado: C:\Users\crisv\OneDrive\Desktop\asistente-lectura-discapacidad\data\processed\textos_embeddings.csv
✓ Modelo guardado: C:\Users\crisv\OneDrive\Desktop\asistente-lectura-discapacidad\models\clasificador_embeddings.pkl


## 10. Probar con Textos Nuevos

In [11]:
textos_prueba = [
    "El perro juega en el parque. Es un perro grande y feliz. Le gusta correr.",
    "La economía española presenta indicadores positivos según los últimos datos publicados por el Banco de España.",
    "El proceso de fotosíntesis implica la transformación de energía lumínica en energía química mediante complejos mecanismos bioquímicos que tienen lugar en los cloroplastos.",
    "El Real Decreto 1234/2024 establece las bases reguladoras para la concesión de subvenciones destinadas a entidades sin ánimo de lucro.",
    "Para preparar el risotto, primero hay que tostar el arroz carnaroli en mantequilla clarificada hasta alcanzar una temperatura óptima de 180°C."
]

niveles_esperados = [1, 2, 3, 4, 5]

print("\n" + "=" * 80)
print("PREDICCIONES EN TEXTOS DE PRUEBA")
print("=" * 80 + "\n")

# Generar embeddings de textos de prueba
embeddings_prueba = embedding_model.encode(textos_prueba)

# Calcular features lingüísticas para textos de prueba
features_prueba = pd.DataFrame([calcular_features_adicionales(t) for t in textos_prueba])
features_prueba_norm = scaler.transform(features_prueba)

# Combinar embeddings + features
X_prueba_hibrido = np.hstack([embeddings_prueba, features_prueba_norm])

# Predecir (XGBoost da clases 0-4)
predicciones_xgb = clf.predict(X_prueba_hibrido)
probabilidades = clf.predict_proba(X_prueba_hibrido)

# Convertir a niveles reales [1,2,3,4,5]
predicciones = predicciones_xgb + 1

for i, (texto, pred, proba, esperado) in enumerate(zip(textos_prueba, predicciones, probabilidades, niveles_esperados), 1):
    acierto = "✓" if pred == esperado else "✗"
    confianza = max(proba) * 100
    
    print(f"Texto {i}: {acierto}")
    print(f"  '{texto[:80]}...'")
    print(f"  → Esperado: Nivel {esperado}")
    print(f"  → Predicho: Nivel {pred}")


PREDICCIONES EN TEXTOS DE PRUEBA

Texto 1: ✓
  'El perro juega en el parque. Es un perro grande y feliz. Le gusta correr....'
  → Esperado: Nivel 1
  → Predicho: Nivel 1
Texto 2: ✗
  'La economía española presenta indicadores positivos según los últimos datos publ...'
  → Esperado: Nivel 2
  → Predicho: Nivel 1
Texto 3: ✓
  'El proceso de fotosíntesis implica la transformación de energía lumínica en ener...'
  → Esperado: Nivel 3
  → Predicho: Nivel 3
Texto 4: ✗
  'El Real Decreto 1234/2024 establece las bases reguladoras para la concesión de s...'
  → Esperado: Nivel 4
  → Predicho: Nivel 1
Texto 5: ✗
  'Para preparar el risotto, primero hay que tostar el arroz carnaroli en mantequil...'
  → Esperado: Nivel 5
  → Predicho: Nivel 1


## ✅ RESUMEN FINAL

In [12]:
print("\n" + "=" * 80)
print("✓ PROCESAMIENTO COMPLETADO CON ÉXITO (EMBEDDINGS)")
print("=" * 80)
print(f"\n📊 ESTADÍSTICAS FINALES:")
print(f"  - Dataset: {len(df)} textos")
print(f"  - Distribución: {dict(df['nivel'].value_counts().sort_index())}")
print(f"  - CSV: {OUTPUT_CSV}")
print(f"\n🤖 MODELO:")
print(f"  - Método: Sentence Transformers + XGBoost")
print(f"  - Dimensiones: 384 (embeddings)")
print(f"  - Accuracy (test): {accuracy:.2%}")
print(f"  - Accuracy (CV): {cv_scores.mean():.2%} ± {cv_scores.std():.2%}")
print(f"  - Modelo guardado: {OUTPUT_MODEL}")
print(f"\n🎯 PRÓXIMOS PASOS:")
print(f"  1. Dashboard con visualizaciones")
print(f"  2. Clustering K-means de usuarios")
print(f"  3. Sistema RAG (opcional)")
print(f"  4. Piloto con usuarios reales")
print("\n" + "=" * 80)


✓ PROCESAMIENTO COMPLETADO CON ÉXITO (EMBEDDINGS)

📊 ESTADÍSTICAS FINALES:
  - Dataset: 2695 textos
  - Distribución: {1: np.int64(380), 2: np.int64(760), 3: np.int64(651), 4: np.int64(466), 5: np.int64(438)}
  - CSV: C:\Users\crisv\OneDrive\Desktop\asistente-lectura-discapacidad\data\processed\textos_embeddings.csv

🤖 MODELO:
  - Método: Sentence Transformers + XGBoost
  - Dimensiones: 384 (embeddings)
  - Accuracy (test): 90.54%
  - Accuracy (CV): 82.82% ± 7.85%
  - Modelo guardado: C:\Users\crisv\OneDrive\Desktop\asistente-lectura-discapacidad\models\clasificador_embeddings.pkl

🎯 PRÓXIMOS PASOS:
  1. Dashboard con visualizaciones
  2. Clustering K-means de usuarios
  3. Sistema RAG (opcional)
  4. Piloto con usuarios reales



In [13]:
#!/usr/bin/env python3
"""
Análisis N3: ¿Qué textos se clasifican mal y por qué?
"""

# EJECUTAR EN JUPYTER, en celda nueva después del entrenamiento

print("=" * 80)
print("ANÁLISIS NIVEL 3 - ¿QUÉ FALLA?")
print("=" * 80)

import pandas as pd

# Usar variables del entrenamiento
def calcular_complejidad(texto):
    palabras = len(texto.split())
    oraciones = max(1, texto.count('.') + texto.count('!') + texto.count('?'))
    palabras_por_oracion = palabras / oraciones
    letras_por_palabra = len(texto.replace(' ', '')) / max(1, palabras)
    return palabras_por_oracion * 10 + letras_por_palabra * 20

# Cargar datos completos
df_original = pd.read_csv(r"C:\Users\crisv\OneDrive\Desktop\asistente-lectura-discapacidad\data\processed\textos_embeddings.csv")

# Split igual que en entrenamiento
from sklearn.model_selection import train_test_split
_, df_test = train_test_split(df_original, test_size=0.2, random_state=42, stratify=df_original['nivel'])

# Calcular complejidad
df_test['complejidad'] = df_test['texto'].apply(calcular_complejidad)
df_test['prediccion'] = y_pred

# ANÁLISIS N3
print("\n" + "=" * 80)
print("ANÁLISIS NIVEL 3")
print("=" * 80)

n3_real = df_test[df_test['nivel'] == 3].copy()
n3_correcto = n3_real[n3_real['prediccion'] == 3]
n3_como_n2 = n3_real[n3_real['prediccion'] == 2]
n3_como_n4 = n3_real[n3_real['prediccion'] == 4]
n3_otros = n3_real[~n3_real['prediccion'].isin([2, 3, 4])]

print(f"\nN3 total: {len(n3_real)} fragmentos")
print(f"  ✓ Correcto (N3→N3): {len(n3_correcto)} ({len(n3_correcto)/len(n3_real)*100:.1f}%)")
print(f"  ✗ Confusión N3→N2: {len(n3_como_n2)} ({len(n3_como_n2)/len(n3_real)*100:.1f}%)")
print(f"  ✗ Confusión N3→N4: {len(n3_como_n4)} ({len(n3_como_n4)/len(n3_real)*100:.1f}%)")
print(f"  ✗ Otros: {len(n3_otros)} ({len(n3_otros)/len(n3_real)*100:.1f}%)")

# COMPLEJIDAD
print("\n" + "=" * 80)
print("COMPLEJIDAD DE TEXTOS N3")
print("=" * 80)

print(f"\nN3 correctamente clasificados:")
print(f"  Complejidad media: {n3_correcto['complejidad'].mean():.1f}")
print(f"  Rango: {n3_correcto['complejidad'].min():.1f} - {n3_correcto['complejidad'].max():.1f}")

if len(n3_como_n2) > 0:
    print(f"\nN3 clasificados como N2 (MUY SIMPLES):")
    print(f"  Complejidad media: {n3_como_n2['complejidad'].mean():.1f}")
    print(f"  Rango: {n3_como_n2['complejidad'].min():.1f} - {n3_como_n2['complejidad'].max():.1f}")
    diferencia_n2 = n3_correcto['complejidad'].mean() - n3_como_n2['complejidad'].mean()
    print(f"  📊 Son {diferencia_n2:.1f} puntos MÁS SIMPLES que N3 correcto")

if len(n3_como_n4) > 0:
    print(f"\nN3 clasificados como N4 (MUY COMPLEJOS):")
    print(f"  Complejidad media: {n3_como_n4['complejidad'].mean():.1f}")
    print(f"  Rango: {n3_como_n4['complejidad'].min():.1f} - {n3_como_n4['complejidad'].max():.1f}")
    diferencia_n4 = n3_como_n4['complejidad'].mean() - n3_correcto['complejidad'].mean()
    print(f"  📊 Son {diferencia_n4:.1f} puntos MÁS COMPLEJOS que N3 correcto")

# ARCHIVOS PROBLEMÁTICOS
print("\n" + "=" * 80)
print("ARCHIVOS N3 QUE SE CLASIFICAN COMO N2 (MUY SIMPLES)")
print("=" * 80)

if len(n3_como_n2) > 0:
    # Agrupar por archivo si existe la columna
    if 'archivo' in n3_como_n2.columns:
        archivos_n2 = n3_como_n2.groupby('archivo').agg({
            'texto': 'count',
            'complejidad': 'mean'
        }).sort_values('texto', ascending=False)
        
        print(f"\nArchivos que se confunden con N2:")
        for archivo, row in archivos_n2.head(10).iterrows():
            print(f"  {archivo}: {int(row['texto'])} frag, complejidad {row['complejidad']:.1f}")
    else:
        print(f"\n{len(n3_como_n2)} fragmentos clasificados como N2")
        print(f"Complejidad media: {n3_como_n2['complejidad'].mean():.1f}")

print("\n" + "=" * 80)
print("ARCHIVOS N3 QUE SE CLASIFICAN COMO N4 (MUY COMPLEJOS)")
print("=" * 80)

if len(n3_como_n4) > 0:
    if 'archivo' in n3_como_n4.columns:
        archivos_n4 = n3_como_n4.groupby('archivo').agg({
            'texto': 'count',
            'complejidad': 'mean'
        }).sort_values('texto', ascending=False)
        
        print(f"\nArchivos que se confunden con N4:")
        for archivo, row in archivos_n4.head(10).iterrows():
            print(f"  {archivo}: {int(row['texto'])} frag, complejidad {row['complejidad']:.1f}")
    else:
        print(f"\n{len(n3_como_n4)} fragmentos clasificados como N4")
        print(f"Complejidad media: {n3_como_n4['complejidad'].mean():.1f}")

# ESTADÍSTICAS DATASET COMPLETO N3
print("\n" + "=" * 80)
print("DATASET COMPLETO N3 (todos los fragmentos, no solo test)")
print("=" * 80)

n3_completo = df_original[df_original['nivel'] == 3].copy()
n3_completo['complejidad'] = n3_completo['texto'].apply(calcular_complejidad)

print(f"\nTotal fragmentos N3: {len(n3_completo)}")
print(f"Complejidad media: {n3_completo['complejidad'].mean():.1f}")
print(f"Rango: {n3_completo['complejidad'].min():.1f} - {n3_completo['complejidad'].max():.1f}")
print(f"Desviación estándar: {n3_completo['complejidad'].std():.1f}")

# CONCLUSIÓN
print("\n" + "=" * 80)
print("RECOMENDACIÓN")
print("=" * 80)

errores_n2 = len(n3_como_n2)
errores_n4 = len(n3_como_n4)

if errores_n2 > errores_n4:
    print(f"""
N3 tiene más confusión con N2 ({errores_n2} vs {errores_n4} con N4)

PROBLEMA: Textos N3 demasiado SIMPLES
SOLUCIÓN: Añadir 10-15 textos N3 MÁS COMPLEJOS (complejidad 420-450)

BUSCAR:
- National Geographic Kids artículos largos
- Divulgación científica juvenil
- Artículos educativos complejos
""")
elif errores_n4 > errores_n2:
    print(f"""
N3 tiene más confusión con N4 ({errores_n4} vs {errores_n2} con N2)

PROBLEMA: Textos N3 demasiado COMPLEJOS
SOLUCIÓN: Añadir 10-15 textos N3 MÁS SIMPLES (complejidad 380-410)

BUSCAR:
- National Geographic Kids artículos cortos
- Cuentos juveniles sencillos
- Noticias adaptadas
""")
else:
    print(f"""
N3 tiene confusión BALANCEADA entre N2 y N4

SOLUCIÓN: Añadir 10-15 textos N3 en la ZONA MEDIA (complejidad 400-420)
Para reforzar el centro de N3
""")

print("=" * 80)

ANÁLISIS NIVEL 3 - ¿QUÉ FALLA?

ANÁLISIS NIVEL 3

N3 total: 130 fragmentos
  ✓ Correcto (N3→N3): 117 (90.0%)
  ✗ Confusión N3→N2: 8 (6.2%)
  ✗ Confusión N3→N4: 3 (2.3%)
  ✗ Otros: 2 (1.5%)

COMPLEJIDAD DE TEXTOS N3

N3 correctamente clasificados:
  Complejidad media: 460.6
  Rango: 340.8 - 810.6

N3 clasificados como N2 (MUY SIMPLES):
  Complejidad media: 401.6
  Rango: 358.9 - 471.1
  📊 Son 58.9 puntos MÁS SIMPLES que N3 correcto

N3 clasificados como N4 (MUY COMPLEJOS):
  Complejidad media: 467.9
  Rango: 421.0 - 546.9
  📊 Son 7.3 puntos MÁS COMPLEJOS que N3 correcto

ARCHIVOS N3 QUE SE CLASIFICAN COMO N2 (MUY SIMPLES)

8 fragmentos clasificados como N2
Complejidad media: 401.6

ARCHIVOS N3 QUE SE CLASIFICAN COMO N4 (MUY COMPLEJOS)

3 fragmentos clasificados como N4
Complejidad media: 467.9

DATASET COMPLETO N3 (todos los fragmentos, no solo test)

Total fragmentos N3: 651
Complejidad media: 456.1
Rango: 292.0 - 821.4
Desviación estándar: 85.4

RECOMENDACIÓN

N3 tiene más confusión c

In [14]:
#!/usr/bin/env python3
"""
Análisis detallado de archivos N3 problemáticos
"""

import pandas as pd
from pathlib import Path

# Configuración
BASE_DIR = Path(r"C:\Users\crisv\OneDrive\Desktop\asistente-lectura-discapacidad")
CSV_PATH = BASE_DIR / "data" / "processed" / "textos_embeddings.csv"

print("=" * 80)
print("ANÁLISIS DETALLADO DE ARCHIVOS N3")
print("=" * 80)

# Cargar datos completos
df = pd.read_csv(CSV_PATH)
print(f"\n📊 Dataset: {len(df)} fragmentos totales")

# Verificar columnas
print(f"Columnas disponibles: {list(df.columns)[:10]}...")

# Filtrar N3
n3 = df[df['nivel'] == 3].copy()
print(f"\n📊 Nivel 3: {len(n3)} fragmentos")

# Calcular complejidad
def calcular_complejidad(texto):
    palabras = len(texto.split())
    oraciones = max(1, texto.count('.') + texto.count('!') + texto.count('?'))
    palabras_por_oracion = palabras / oraciones
    letras_por_palabra = len(texto.replace(' ', '')) / max(1, palabras)
    return palabras_por_oracion * 10 + letras_por_palabra * 20

n3['complejidad'] = n3['texto'].apply(calcular_complejidad)

# Verificar si existe columna 'archivo'
if 'archivo' in n3.columns:
    print("\n" + "=" * 80)
    print("ANÁLISIS POR ARCHIVO")
    print("=" * 80)
    
    # Agrupar por archivo
    archivos_stats = n3.groupby('archivo').agg({
        'texto': 'count',
        'complejidad': 'mean'
    }).sort_values('complejidad', ascending=True)
    
    print(f"\n{'Archivo':<50} {'Fragmentos':<12} {'Complejidad'}")
    print("-" * 80)
    
    # Top 15 más simples (candidatos a eliminar o revisar)
    print("\n🔴 TOP 15 ARCHIVOS MÁS SIMPLES (posibles problemáticos):")
    print("-" * 80)
    for archivo, row in archivos_stats.head(15).iterrows():
        print(f"{archivo[:47]:<50} {int(row['texto']):<12} {row['complejidad']:.1f}")
    
    # Top 10 más complejos (buenos ejemplos)
    print("\n✅ TOP 10 ARCHIVOS MÁS COMPLEJOS (buenos ejemplos):")
    print("-" * 80)
    for archivo, row in archivos_stats.tail(10).iterrows():
        print(f"{archivo[:47]:<50} {int(row['texto']):<12} {row['complejidad']:.1f}")
    
    # Estadísticas
    print("\n" + "=" * 80)
    print("ESTADÍSTICAS DE COMPLEJIDAD POR ARCHIVO")
    print("=" * 80)
    
    archivos_muy_simples = archivos_stats[archivos_stats['complejidad'] < 420]
    archivos_simples = archivos_stats[(archivos_stats['complejidad'] >= 420) & (archivos_stats['complejidad'] < 450)]
    archivos_medio = archivos_stats[(archivos_stats['complejidad'] >= 450) & (archivos_stats['complejidad'] < 480)]
    archivos_complejos = archivos_stats[archivos_stats['complejidad'] >= 480]
    
    print(f"\nMuy simples (<420): {len(archivos_muy_simples)} archivos")
    print(f"Simples (420-450): {len(archivos_simples)} archivos")
    print(f"Medio (450-480): {len(archivos_medio)} archivos")
    print(f"Complejos (>480): {len(archivos_complejos)} archivos")
    
    # RECOMENDACIONES
    print("\n" + "=" * 80)
    print("RECOMENDACIONES")
    print("=" * 80)
    
    if len(archivos_muy_simples) > 0:
        print(f"""
⚠️  ARCHIVOS MUY SIMPLES DETECTADOS ({len(archivos_muy_simples)} archivos)

Estos archivos tienen complejidad <420 y probablemente se confunden con N2:
""")
        for archivo, row in archivos_muy_simples.head(10).iterrows():
            print(f"  • {archivo}: {row['complejidad']:.1f} puntos")
        
        print(f"""
OPCIONES:
1. ELIMINAR estos {len(archivos_muy_simples)} archivos de N3
   (moverlos a N2 o descartarlos)
   
2. AÑADIR 12-15 archivos MÁS COMPLEJOS (450-480 puntos)
   para balancear
   
RECOMENDADO: Opción 2 (añadir textos complejos)
""")
    
    if len(archivos_complejos) < 5:
        print(f"""
⚠️  POCOS ARCHIVOS COMPLEJOS ({len(archivos_complejos)} archivos)

N3 necesita más ejemplos en el rango 480+ para evitar confusión con N4

BUSCAR:
- National Geographic Kids artículos científicos largos
- Divulgación científica juvenil
- Artículos educativos complejos
""")

else:
    print("\n⚠️  No se encontró columna 'archivo' en el CSV")
    print("El CSV solo tiene estas columnas:", df.columns.tolist())
    
    # Análisis alternativo por rangos de complejidad
    print("\n" + "=" * 80)
    print("DISTRIBUCIÓN DE COMPLEJIDAD N3")
    print("=" * 80)
    
    muy_simples = n3[n3['complejidad'] < 420]
    simples = n3[(n3['complejidad'] >= 420) & (n3['complejidad'] < 450)]
    medio = n3[(n3['complejidad'] >= 450) & (n3['complejidad'] < 480)]
    complejos = n3[n3['complejidad'] >= 480]
    
    print(f"\nMuy simples (<420): {len(muy_simples)} fragmentos ({len(muy_simples)/len(n3)*100:.1f}%)")
    print(f"Simples (420-450): {len(simples)} fragmentos ({len(simples)/len(n3)*100:.1f}%)")
    print(f"Medio (450-480): {len(medio)} fragmentos ({len(medio)/len(n3)*100:.1f}%)")
    print(f"Complejos (>480): {len(complejos)} fragmentos ({len(complejos)/len(n3)*100:.1f}%)")
    
    print(f"""
RECOMENDACIÓN:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Tienes {len(muy_simples)} fragmentos muy simples (<420)
que probablemente se confunden con N2

SOLUCIÓN: Añadir 12-15 textos en rango 450-480 puntos
para subir la media y reducir confusión con N2
""")

print("\n" + "=" * 80)

ANÁLISIS DETALLADO DE ARCHIVOS N3

📊 Dataset: 2695 fragmentos totales
Columnas disponibles: ['texto', 'nivel', 'fuente', 'fragmento_num']...

📊 Nivel 3: 651 fragmentos

⚠️  No se encontró columna 'archivo' en el CSV
El CSV solo tiene estas columnas: ['texto', 'nivel', 'fuente', 'fragmento_num']

DISTRIBUCIÓN DE COMPLEJIDAD N3

Muy simples (<420): 258 fragmentos (39.6%)
Simples (420-450): 94 fragmentos (14.4%)
Medio (450-480): 70 fragmentos (10.8%)
Complejos (>480): 229 fragmentos (35.2%)

RECOMENDACIÓN:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Tienes 258 fragmentos muy simples (<420)
que probablemente se confunden con N2

SOLUCIÓN: Añadir 12-15 textos en rango 450-480 puntos
para subir la media y reducir confusión con N2




In [15]:
#!/usr/bin/env python3
"""
Análisis de fuentes N3 (archivos origen)
"""

import pandas as pd
from pathlib import Path

BASE_DIR = Path(r"C:\Users\crisv\OneDrive\Desktop\asistente-lectura-discapacidad")
CSV_PATH = BASE_DIR / "data" / "processed" / "textos_embeddings.csv"

print("=" * 80)
print("ANÁLISIS DE FUENTES N3")
print("=" * 80)

df = pd.read_csv(CSV_PATH)
n3 = df[df['nivel'] == 3].copy()

def calcular_complejidad(texto):
    palabras = len(texto.split())
    oraciones = max(1, texto.count('.') + texto.count('!') + texto.count('?'))
    palabras_por_oracion = palabras / oraciones
    letras_por_palabra = len(texto.replace(' ', '')) / max(1, palabras)
    return palabras_por_oracion * 10 + letras_por_palabra * 20

n3['complejidad'] = n3['texto'].apply(calcular_complejidad)

# Usar columna 'fuente'
if 'fuente' in n3.columns:
    print("\n" + "=" * 80)
    print("ARCHIVOS N3 POR COMPLEJIDAD")
    print("=" * 80)
    
    fuentes_stats = n3.groupby('fuente').agg({
        'texto': 'count',
        'complejidad': 'mean'
    }).sort_values('complejidad', ascending=True)
    
    # TOP 20 MÁS SIMPLES
    print("\n🔴 TOP 20 ARCHIVOS MÁS SIMPLES (<440 puntos):")
    print("-" * 80)
    print(f"{'Archivo':<60} {'Frag':<8} {'Complejidad'}")
    print("-" * 80)
    
    simples = fuentes_stats[fuentes_stats['complejidad'] < 440].head(20)
    for archivo, row in simples.iterrows():
        print(f"{str(archivo)[:57]:<60} {int(row['texto']):<8} {row['complejidad']:.1f}")
    
    # TOP 15 MÁS COMPLEJOS
    print("\n✅ TOP 15 ARCHIVOS MÁS COMPLEJOS (>450 puntos):")
    print("-" * 80)
    print(f"{'Archivo':<60} {'Frag':<8} {'Complejidad'}")
    print("-" * 80)
    
    complejos = fuentes_stats[fuentes_stats['complejidad'] > 450].tail(15)
    for archivo, row in complejos.iterrows():
        print(f"{str(archivo)[:57]:<60} {int(row['texto']):<8} {row['complejidad']:.1f}")
    
    # ANÁLISIS
    print("\n" + "=" * 80)
    print("DECISIÓN: ¿ELIMINAR O AÑADIR?")
    print("=" * 80)
    
    muy_simples = fuentes_stats[fuentes_stats['complejidad'] < 420]
    
    print(f"""
ARCHIVOS MUY SIMPLES: {len(muy_simples)}
Total fragmentos en estos archivos: {muy_simples['texto'].sum()}

OPCIONES:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

OPCIÓN A - ELIMINAR ARCHIVOS SIMPLES:
✂️  Eliminar los {len(muy_simples)} archivos más simples
    Reduce dataset N3: 558 → {558 - muy_simples['texto'].sum()} fragmentos
    Tiempo: 20 min
    Resultado esperado: N3: 77% → 83-85%

OPCIÓN B - AÑADIR ARCHIVOS COMPLEJOS:
➕ Añadir 12-15 archivos nuevos (450-480 puntos)
    Aumenta dataset N3: 558 → ~730 fragmentos
    Tiempo: 2 horas
    Resultado esperado: N3: 77% → 85-88%

OPCIÓN C - COMBINACIÓN (RECOMENDADA):
✂️➕ Eliminar 5 archivos más simples + Añadir 10 complejos
    Tiempo: 1.5 horas
    Resultado esperado: N3: 77% → 88-90% 🏆
""")

print("\n" + "=" * 80)

ANÁLISIS DE FUENTES N3

ARCHIVOS N3 POR COMPLEJIDAD

🔴 TOP 20 ARCHIVOS MÁS SIMPLES (<440 puntos):
--------------------------------------------------------------------------------
Archivo                                                      Frag     Complejidad
--------------------------------------------------------------------------------
nat_geo_antiguo_egipto.txt                                   47       425.4
nat_geo_errores_cientificos.txt                              23       434.1
nat_geo_exploradores.txt                                     35       437.2
nat_geo_pinguino_adelia.txt                                  9        439.5
nat_geo_pinguino_barbijo.txt                                 23       439.7
nat_geo_ualabi.txt                                           13       439.8

✅ TOP 15 ARCHIVOS MÁS COMPLEJOS (>450 puntos):
--------------------------------------------------------------------------------
Archivo                                                      Frag     Com